In [ ]:
# === XGBoost + SHAP Beeswarm + Global Importance Bars ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import shap
from scipy.stats import gaussian_kde
import xgboost as xgb
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
DATA_PATH = r"D:\研究\ABF新\8.13ML\ML\生产率及解释变量合并2.xlsx"
TARGET = "kcal_per_kg"   # or "gProtein_per_kg"
FEATURES = [
    "Price_Change","Export_Orientation","Import_Dependence_Ratio",
    "ABF_Consumption_Percapita","SSR","Pr","T","GDP",
    "Ruminant_Share","Poultry_Share","Feed_Yield","Feed_Price",
    "Stocking_Density","Slaughter_Efficiency"
]

# 新增输出路径配置
OUTPUT_PATH = r"D:\研究\ABF新\论文的图\SHAP_beeswarm_plot.png"

# 字体配置
FONT_CONFIG = {
    'family': 'Arial',
    'size': 12
}
plt.rc('font', **FONT_CONFIG)

TOP_K = 20
MAX_POINTS = 2000
POINT_SIZE = 12
POINT_ALPHA = 0.85
POINT_EDGE = 0.25
MAX_SPREAD = 0.28
KDE_BW = "scott"
CMAP_POINTS = plt.get_cmap("viridis")
BAR_ALPHA = 0.35
GRID_ALPHA = 0.15
FIGSIZE = (8.5, 8)  # 稍微增大图形尺寸以适应更大字体

# ---------------- Load Data ----------------
df = pd.read_excel(DATA_PATH)
X = df[FEATURES].copy()
y = df[TARGET].copy()

# handle missing values
X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# ---------------- Train/test split ----------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------- Train XGBoost ----------------
model = xgb.XGBRegressor(
    random_state=42, tree_method="hist",
    objective="reg:squarederror", eval_metric="rmse",
    n_estimators=800, learning_rate=0.05, max_depth=6
)
model.fit(X_train, y_train)

# ---------------- SHAP values ----------------
explainer = shap.TreeExplainer(model)
X_for_shap = X_test.copy()
if len(X_for_shap) > MAX_POINTS:
    X_for_shap = X_for_shap.sample(n=MAX_POINTS, random_state=42)

shap_values = explainer.shap_values(X_for_shap)

# convert to 2D if needed
def to_2d(shap_vals):
    if isinstance(shap_vals, list):
        arrs = [np.abs(a) for a in shap_vals]
        return np.mean(np.stack(arrs, axis=0), axis=0)
    elif isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        return np.mean(np.abs(shap_vals), axis=2)
    else:
        return shap_vals

shap_2d = to_2d(shap_values)
mean_abs = np.mean(np.abs(shap_2d), axis=0)

# ---- Sort features ----
desc_idx = np.argsort(mean_abs)[::-1]
feat_desc = [X_train.columns[i] for i in desc_idx][:TOP_K]
mean_desc = mean_abs[desc_idx][:TOP_K]

feat_plot = feat_desc[::-1]   # least important bottom, most important top
mean_plot = mean_desc[::-1]
name2idx = {n: i for i, n in enumerate(X_train.columns)}
col_idx_plot = [name2idx[f] for f in feat_plot]
shap_ord = shap_2d[:, col_idx_plot]
X_vals_ord = X_for_shap[feat_plot].to_numpy()

# ---------------- 格式化特征名称 ----------------
def format_feature_name(name):
    """将特征名称中的下划线替换为空格并适当格式化"""
    replacements = {
        'ABF_Consumption_Percapita': 'ABF Consumption Per Capita',
        'Import_Dependence_Ratio': 'Import Dependence Ratio',
        'Export_Orientation': 'Export Orientation',
        'Price_Change': 'Price Change',
        'Ruminant_Share': 'Ruminant Share',
        'Poultry_Share': 'Poultry Share',
        'Feed_Yield': 'Feed Yield',
        'Feed_Price': 'Feed Price',
        'Stocking_Density': 'Stocking Density',
        'Slaughter_Efficiency': 'Slaughter Efficiency',
        'GDP': 'GDP',
        'SSR': 'SSR',
        'Pr': 'Precipitation',
        'T': 'Temperature'
    }
    return replacements.get(name, name.replace('_', ' ').title())

formatted_feat_plot = [format_feature_name(f) for f in feat_plot]

# ---------------- Density jitter ----------------
def density_beeswarm_offsets(x, max_spread=0.28, bw_method="scott"):
    x = np.asarray(x, dtype=float)
    mask = ~np.isnan(x)
    if np.sum(mask) < 5 or len(np.unique(x[mask])) < 5:
        rng = np.random.default_rng(42)
        return (rng.random(len(x)) - 0.5) * 2 * (0.08 * max_spread)
    kde = gaussian_kde(x[mask], bw_method=bw_method)
    dens = np.zeros_like(x, dtype=float)
    dens[mask] = kde.evaluate(x[mask])
    dmax = dens[mask].max()
    width = max_spread * (dens / dmax if dmax > 0 else 0.0)
    rng = np.random.default_rng(42)
    jitter = (rng.random(len(x)) - 0.5) * 2 * width
    return jitter

# ---------------- Plotting ----------------
fig = plt.figure(figsize=FIGSIZE)
ax = plt.gca()

ax.axvline(0, color="#9a9a9a", lw=1.0, zorder=0)
ax.xaxis.grid(True, which="major", linestyle="--", alpha=GRID_ALPHA)

# beeswarm points
for row in range(len(formatted_feat_plot)):
    xs = shap_ord[:, row]
    vals = X_vals_ord[:, row]
    vmin, vmax = np.nanpercentile(vals, 5), np.nanpercentile(vals, 95)
    norm = Normalize(vmin=vmin, vmax=vmax)
    colors = CMAP_POINTS(norm(vals))
    yj = row + density_beeswarm_offsets(xs, max_spread=MAX_SPREAD, bw_method=KDE_BW)
    ax.scatter(xs, yj, s=POINT_SIZE, alpha=POINT_ALPHA,
               c=colors, edgecolors="black", linewidths=POINT_EDGE, zorder=3)

ax.set_yticks(np.arange(len(formatted_feat_plot)))
ax.set_yticklabels(formatted_feat_plot, fontsize=16)  # 增大y轴标签字体
ax.set_xlabel("SHAP value", fontsize=18)  # 增大x轴标签字体

xmin = np.nanpercentile(shap_ord, 0.5)
xmax = np.nanpercentile(shap_ord, 99.5)
rng = xmax - xmin
ax.set_xlim(xmin - 0.06*rng, xmax + 0.06*rng)

# 设置x轴刻度字体大小
ax.tick_params(axis='x', labelsize=16)

# global importance bars
ax_top = ax.twiny()
ax_top.set_xlim(0, float(mean_plot.max()) * 1.05)
for y0, v in enumerate(mean_plot):
    ax_top.barh(y0, v, height=0.78, color="#cfcfcf", alpha=BAR_ALPHA,
                edgecolor="none", zorder=1)
ax_top.set_xlabel("Mean |SHAP| (Global Importance)", fontsize=18)  # 增大标签字体
ax_top.set_xticks(np.linspace(0, float(mean_plot.max())*1.05, 5))
# 设置顶部x轴刻度字体大小
ax_top.tick_params(axis='x', labelsize=16)

# colorbar
fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.88, 0.15, 0.02, 0.70])
cb = fig.colorbar(ScalarMappable(norm=Normalize(0,1), cmap=CMAP_POINTS), cax=cax)
cb.set_label("Feature value", fontsize=16)  # 增大colorbar标签字体
cb.ax.tick_params(labelsize=12)  # 增大colorbar刻度字体
cb.ax.text(3, 1, "High", transform=cb.ax.transAxes, fontsize=14, va="center")  # 增大字体
cb.ax.text(3, 0, "Low",  transform=cb.ax.transAxes, fontsize=14, va="center")  # 增大字体

# legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Samples (colored by feature value)',
           markerfacecolor='#7f7f7f', markeredgecolor='black',
           markersize=8, linestyle='none'),  # 增大标记尺寸
    Patch(facecolor='#cfcfcf', alpha=BAR_ALPHA,
          edgecolor='none', label='Global importance)'),
]
ax.legend(handles=legend_elements, loc='lower right',
          frameon=True, framealpha=0.8, facecolor='white', edgecolor='none',
          fontsize=11)  # 增大图例字体

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout(rect=[0.0, 0.0, 0.88, 1.0])

# ---------------- 保存图片 ----------------
plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight', facecolor='white')
print(f"图片已保存至: {OUTPUT_PATH}")

plt.show()

In [ ]:
# === SHAP 前六依赖图（带去尾/忽略异常值） ===
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# -------- 配置（新增去尾参数） --------
TOP_M = 6                 # 画前六
N_ROWS, N_COLS = 3, 2
POINT_SIZE = POINT_SIZE        # 与上面保持一致（你前面定义了）
POINT_ALPHA = POINT_ALPHA
SMOOTH_FRAC = 0.20        # 平滑比例，0.15~0.3 合理
CMAP = plt.get_cmap("plasma") # 与 beeswarm 使用同一配色
FIGSIZE_DEP = (8.5, 8)

# —— 去尾参数（可改）——
CLIP_PCT_X = (1, 99)      # 特征值分位剪裁
CLIP_PCT_Y = (1, 99)      # SHAP值分位剪裁
IQR_K_X = None            # 若想用 IQR 规则裁剪 X，设为 3.0 等；None 不启用
IQR_K_Y = 3.0             # 对 SHAP 值的 IQR 规则（更稳健）；设 None 则不启用
MIN_AFTER_TRIM = 50       # 去尾后至少保留的最小点数，不足则自动放宽为“不过滤”

# 可读性重命名（按你这批特征）
FEATURE_RENAME = {
    "Price_Change": "Price change",
    "Export_Orientation": "Export orientation",
    "Import_Dependence_Ratio": "Import dependence",
    "ABF_Consumption_Percapita": "ABF per capita",
    "SSR": "SSR",
    "Pr": "Precipitation",
    "T": "Temperature",
    "GDP": "GDP",
    "Ruminant_Share": "Ruminant %",
    "Poultry_Share": "Poultry %",
    "Feed_Yield": "Feed yield",
    "Feed_Price": "Feed price",
    "Stocking_Density": "Stocking density",
    "Slaughter_Efficiency": "Slaughter efficiency",
}

# --------- 工具函数 ---------
def smooth_xy(x, y, frac=0.2, min_win=5):
    """简洁平滑曲线（移动平均），无需额外依赖。"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = (~np.isnan(x)) & (~np.isnan(y)) & np.isfinite(x) & np.isfinite(y)
    if m.sum() < 10:
        xs, order = np.unique(x[m], return_index=True)
        return xs, y[m][order][:len(xs)]
    xs = x[m]
    ys = y[m]
    order = np.argsort(xs)
    xs = xs[order]
    ys = ys[order]
    win = max(min_win, int(len(xs) * frac))
    if win % 2 == 0:
        win += 1
    kernel = np.ones(win, dtype=float) / win
    ys_pad = np.pad(ys, (win//2, win//2), mode='edge')
    ys_smooth = np.convolve(ys_pad, kernel, mode='valid')
    return xs, ys_smooth

def _apply_iqr_clip(arr, k):
    """返回 (lo, hi) IQR 界限，用于裁剪；若 k 为 None 或数据不足则返回 (None, None)。"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if (k is None) or (arr.size < 5):
        return None, None
    q1, q3 = np.nanpercentile(arr, [25, 75])
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    return lo, hi

def _clip_bounds(arr, pct_pair=None, iqr_k=None):
    """综合分位数与 IQR 计算上下界（取交集）。"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    lo, hi = -np.inf, np.inf
    # 分位数界
    if (pct_pair is not None) and (arr.size >= 5):
        p_lo, p_hi = np.nanpercentile(arr, pct_pair)
        lo = max(lo, p_lo)
        hi = min(hi, p_hi)
    # IQR 界
    iqr_lo, iqr_hi = _apply_iqr_clip(arr, iqr_k)
    if iqr_lo is not None:
        lo = max(lo, iqr_lo)
        hi = min(hi, iqr_hi)
    if not np.isfinite(lo):
        lo = None
    if not np.isfinite(hi):
        hi = None
    return lo, hi

def build_trim_mask(x, y,
                    clip_pct_x=CLIP_PCT_X, clip_pct_y=CLIP_PCT_Y,
                    iqr_k_x=IQR_K_X, iqr_k_y=IQR_K_Y,
                    min_after=MIN_AFTER_TRIM):
    """
    生成布尔掩码：同时对 x(特征) 与 y(SHAP) 去尾，取交集。
    若去尾后样本数 < min_after，则返回仅有限值过滤（不去尾）。
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    base = np.isfinite(x) & np.isfinite(y)
    if base.sum() < min_after:
        return base  # 点太少，只做有限值过滤

    x_lo, x_hi = _clip_bounds(x[base], pct_pair=clip_pct_x, iqr_k=iqr_k_x)
    y_lo, y_hi = _clip_bounds(y[base], pct_pair=clip_pct_y, iqr_k=iqr_k_y)

    m = base.copy()
    if x_lo is not None:
        m &= (x >= x_lo)
    if x_hi is not None:
        m &= (x <= x_hi)
    if y_lo is not None:
        m &= (y >= y_lo)
    if y_hi is not None:
        m &= (y <= y_hi)

    if m.sum() < min_after:
        # 放宽：仅保留有限值不过滤，避免样本过少
        return base
    return m

# --------- 取重要性前六特征 ---------
feat_names_sorted = feat_desc[:TOP_M]
X_dep = X_for_shap[feat_names_sorted].copy()
shap_cols_idx = [name2idx[f] for f in feat_names_sorted]
shap_dep = shap_2d[:, shap_cols_idx]  # shape: (n_sample, TOP_M)

# —— 颜色：按 Year 或按特征值 —— 
use_year = ("Year" in df.columns)
if use_year:
    # 尝试与 X_for_shap 行对齐
    try:
        year_color_full = df.loc[X_for_shap.index, "Year"].to_numpy()
        norm_all = Normalize(vmin=np.nanpercentile(year_color_full[np.isfinite(year_color_full)], 1),
                             vmax=np.nanpercentile(year_color_full[np.isfinite(year_color_full)], 99))
        cbar_mapper = ScalarMappable(norm=norm_all, cmap=CMAP); cbar_mapper.set_array([])
    except Exception:
        use_year = False

if not use_year:
    dummy_norm = Normalize(vmin=0.0, vmax=1.0)
    cbar_mapper = ScalarMappable(norm=dummy_norm, cmap=CMAP); cbar_mapper.set_array([])

# --------- 绘图 ---------
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=FIGSIZE_DEP)
axes = np.asarray(axes).ravel()

for i, feat in enumerate(feat_names_sorted):
    ax = axes[i]
    x_vals = X_dep[feat].to_numpy(dtype=float)
    y_vals = shap_dep[:, i].astype(float)

    # 去尾掩码
    mask = build_trim_mask(x_vals, y_vals,
                           clip_pct_x=CLIP_PCT_X, clip_pct_y=CLIP_PCT_Y,
                           iqr_k_x=IQR_K_X, iqr_k_y=IQR_K_Y,
                           min_after=MIN_AFTER_TRIM)

    x_t = x_vals[mask]
    y_t = y_vals[mask]

    # 颜色：优先 Year，否则按该特征自身值
    if use_year:
        colors = CMAP(norm_all(year_color_full[mask]))
    else:
        # 注意：颜色归一化基于“去尾后的 x_t”分布，更贴近当前绘制范围
        if x_t.size >= 5:
            vmin, vmax = np.nanpercentile(x_t, [5, 95])
        else:
            vmin, vmax = np.nanmin(x_t), np.nanmax(x_t)
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = np.nanmin(x_t), np.nanmax(x_t)
        norm_feat = Normalize(vmin=vmin, vmax=vmax)
        colors = CMAP(norm_feat(x_t))

    # 散点
    ax.scatter(x_t, y_t, s=POINT_SIZE, alpha=POINT_ALPHA,
               c=colors, edgecolors="none")

    # 平滑趋势线（基于去尾后的点）
    xs, ys = smooth_xy(x_t, y_t, frac=SMOOTH_FRAC)
    ax.plot(xs, ys, color="black", linewidth=2, label="Smoothed trend")
    ax.legend(loc="lower right",
              bbox_to_anchor=(0.98, 0.02),
              frameon=True, framealpha=0.7, facecolor="white", edgecolor="none")

    # 坐标标签&网格
    disp_name = FEATURE_RENAME.get(feat, feat)
    ax.set_xlabel(disp_name, fontsize=16)
    ax.grid(alpha=0.3, linestyle="--")

    # 适度限制 y 轴范围（分位数，避免残余极端值影响）
    if y_t.size >= 10:
        yl, yh = np.nanpercentile(y_t, [0.5, 99.5])
        pad = (yh - yl) * 0.08 if np.isfinite(yl) and np.isfinite(yh) else None
        if pad is not None and pad > 0:
            ax.set_ylim(yl - pad, yh + pad)

# 隐藏多余子图（若不足 2x3）
for j in range(i + 1, N_ROWS * N_COLS):
    axes[j].axis("off")

# 右侧色条（图外）
fig.subplots_adjust(left=0.10, right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(cbar_mapper, cax=cax)
cb.set_label("Year" if use_year else "Feature value",fontsize=18)

# 统一 y 轴标题
try:
    fig.supylabel("SHAP value", x=0.14,y=0.52, fontsize=16)
except Exception:
    fig.text(0.3, 0.5, "SHAP value", rotation=90, va="center", ha="center", fontsize=18)

# 调整子图间距
plt.subplots_adjust(
    wspace=0.1,    # 水平间距，默认约0.2，增大使图更分开
    hspace=0.2     # 垂直间距，默认约0.2，增大使图更分开
)
plt.tight_layout(rect=[0.10, 0.0, 0.88, 1.0])
# 保存图片
output_path = r"D:\研究\ABF新\论文的图\SHAP_dependence_plots.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"依赖图已保存至: {output_path}")
plt.show()

In [1]:
#GAM 临界探索
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pygam import LinearGAM, s, f
from sklearn.preprocessing import StandardScaler
from scipy import stats

# ==================== 配置 ====================
TOP_FEATURES = 6  # 分析前6个重要特征
GAM_SPLINES = 8   # GAM的样条基数
ALPHA = 0.05      # 显著性水平

# ==================== 基于SHAP值构建GAM ====================

def shap_to_gam_analysis(shap_values, feature_data, feature_names, top_k=TOP_FEATURES):
    """
    将SHAP值转换为GAM分析，识别临界点和响应机制
    """
    results = {}
    
    # 计算特征重要性排序
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    top_indices = np.argsort(mean_abs_shap)[::-1][:top_k]
    top_features = [feature_names[i] for i in top_indices]
    
    print("=== 基于SHAP的GAM临界点分析 ===")
    
    for i, feat_idx in enumerate(top_indices):
        feature_name = feature_names[feat_idx]
        X_feat = feature_data.iloc[:, feat_idx].values.reshape(-1, 1)
        y_shap = shap_values[:, feat_idx]
        
        # 移除缺失值
        mask = ~(np.isnan(X_feat.flatten()) | np.isnan(y_shap))
        X_clean = X_feat[mask]
        y_clean = y_shap[mask]
        
        if len(X_clean) < 20:  # 确保足够的数据点
            continue
            
        # 构建GAM模型
        gam = LinearGAM(s(0, n_splines=GAM_SPLINES)).fit(X_clean, y_clean)
        
        # 模型评估
        r2 = gam.statistics_['pseudo_r2']['explained_deviance']
        
        # 生成预测曲线
        XX = gam.generate_X_grid(term=0, n=100)
        YY = gam.predict(XX)
        YY_std = gam.prediction_intervals(XX, width=0.95)  # 置信区间
        
        # 寻找临界点（导数符号变化点）
        critical_points = find_critical_points(XX, YY)
        
        # 存储结果
        results[feature_name] = {
            'gam': gam,
            'X': X_clean,
            'y_shap': y_clean,
            'XX': XX,
            'YY': YY,
            'YY_std': YY_std,
            'r2': r2,
            'critical_points': critical_points,
            'importance': mean_abs_shap[feat_idx]
        }
        
        # 打印分析结果
        print(f"\n📊 {feature_name} (重要性: {mean_abs_shap[feat_idx]:.1f})")
        print(f"   - GAM解释方差: {r2:.3f}")
        print(f"   - 临界点数量: {len(critical_points)}")
        for j, cp in enumerate(critical_points):
            print(f"     临界点 {j+1}: x = {cp['x']:.3f}, SHAP = {cp['y']:.3f}")
            print(f"     区间影响: {cp['effect_direction']}, 强度: {cp['effect_strength']:.3f}")
    
    return results, top_features

def find_critical_points(XX, YY):
    """
    识别GAM曲线中的临界点和响应模式
    """
    if len(YY) < 3:
        return []
    
    # 计算一阶导数（差分近似）
    dy = np.diff(YY)
    dx = np.diff(XX.flatten())
    derivative = dy / dx
    
    critical_points = []
    
    # 寻找导数符号变化点
    for i in range(1, len(derivative)):
        if (derivative[i-1] * derivative[i] <= 0) and (abs(derivative[i-1]) > 1e-5):
            # 找到临界点
            cp_x = XX[i, 0]
            cp_y = YY[i]
            
            # 确定响应模式
            if i < len(derivative) - 1:
                left_slope = np.mean(derivative[max(0, i-2):i])
                right_slope = np.mean(derivative[i:min(len(derivative), i+2)])
                
                if left_slope > 0 and right_slope < 0:
                    pattern = "峰值点"
                    strength = abs(left_slope) + abs(right_slope)
                elif left_slope < 0 and right_slope > 0:
                    pattern = "谷值点" 
                    strength = abs(left_slope) + abs(right_slope)
                else:
                    pattern = "拐点"
                    strength = abs(derivative[i] - derivative[i-1])
            else:
                pattern = "拐点"
                strength = 0.1
                
            # 确定效应方向
            if cp_y > 0:
                direction = "正向影响"
            else:
                direction = "负向影响"
                
            critical_points.append({
                'x': cp_x,
                'y': cp_y,
                'type': pattern,
                'effect_direction': direction,
                'effect_strength': strength
            })
    
    return critical_points

def plot_gam_analysis(results, top_features, save_path=None):
    """
    绘制GAM分析结果
    """
    n_features = len(top_features)
    n_cols = 2
    n_rows = (n_features + 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if n_features > 1 else [axes]

    # 中文类型到英文的映射，仅用于图例显示（不改动数据本身）
    type_map = {"峰值点": "Peak", "谷值点": "Trough", "拐点": "Inflection"}

    for i, feature_name in enumerate(top_features):
        if feature_name not in results:
            continue

        result = results[feature_name]
        ax = axes[i]

        # 散点图
        ax.scatter(
            result['X'], result['y_shap'],
            alpha=0.3, s=20, color='blue', label='SHAP value'
        )

        # GAM曲线
        ax.plot(
            result['XX'], result['YY'],
            'r-', linewidth=3, label='GAM fit'
        )

        # 置信区间
        ax.fill_between(
            result['XX'].flatten(),
            result['YY_std'][:, 0], result['YY_std'][:, 1],
            alpha=0.3, color='red', label='95% CI'
        )

        # 标记临界点（图例使用英文）
        for cp in result['critical_points']:
            cp_label = f"Critical point ({type_map.get(cp['type'], cp['type'])})"
            ax.axvline(x=cp['x'], color='green', linestyle='--', alpha=0.7)
            ax.plot(cp['x'], cp['y'], 'go', markersize=8, label=cp_label)

        ax.set_xlabel(feature_name, fontsize=12)
        ax.set_ylabel('SHAP value', fontsize=12)
        ax.set_title(f'{feature_name}\nGAM R² = {result["r2"]:.3f}', fontsize=14)
        ax.grid(True, alpha=0.3)

        # 只在第一个图显示图例
        if i == 0:
            ax.legend()

    # 隐藏多余的子图
    for j in range(len(top_features), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"GAM分析图已保存至: {save_path}")

    plt.show()


# ==================== 执行分析 ====================

# 准备数据（使用您之前的SHAP计算结果）
X_for_gam = X_for_shap.copy()
shap_values_gam = shap_2d.copy()

# 运行GAM分析
print("开始GAM临界点分析...")
results, top_features = shap_to_gam_analysis(
    shap_values=shap_values_gam,
    feature_data=X_for_gam,
    feature_names=FEATURES,
    top_k=TOP_FEATURES
)

# 绘制结果
output_path = r"D:\研究\ABF新\论文的图\SHAP_GAM_analysis.png"
plot_gam_analysis(results, top_features, save_path=output_path)

# ==================== 生成分析报告 ====================

def generate_gam_report(results):
    """生成详细的GAM分析报告"""
    print("\n" + "="*60)
    print("            GAM临界响应机制分析报告")
    print("="*60)
    
    for feature_name, result in results.items():
        print(f"\n🔍 特征: {feature_name}")
        print(f"   - SHAP重要性: {result['importance']:.1f}")
        print(f"   - GAM解释度: {result['r2']:.3f}")
        
        if result['critical_points']:
            print("   📈 发现的临界机制:")
            for cp in result['critical_points']:
                print(f"      • 在 {feature_name} = {cp['x']:.2f} 处发现{cp['type']}")
                print(f"        效应: {cp['effect_direction']}, 强度: {cp['effect_strength']:.3f}")
        else:
            print("   📈 未发现明显临界点，关系相对平滑")
        
        # 判断整体影响模式
        mean_shap = np.mean(result['y_shap'])
        if mean_shap > 0:
            overall_effect = "整体正向影响"
        else:
            overall_effect = "整体负向影响"
        print(f"   📊 {overall_effect} (平均SHAP: {mean_shap:.3f})")

# 生成报告
generate_gam_report(results)

NameError: name 'X_for_shap' is not defined

In [ ]:
#分地区建模
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
DATA_PATH = r"D:\研究\ABF新\8.13ML\ML\生产率及解释变量合并2.xlsx"
TARGET = "kcal_per_kg"   # or "gProtein_per_kg"
FEATURES = [
    "Price_Change","Export_Orientation","Import_Dependence_Ratio",
    "ABF_Consumption_Percapita","SSR","Pr","T","GDP",
    "Ruminant_Share","Poultry_Share","Feed_Yield","Feed_Price",
    "Stocking_Density","Slaughter_Efficiency"
]

# ---------------- Load Data ----------------
df = pd.read_excel(DATA_PATH)
X = df[FEATURES].copy()
y = df[TARGET].copy()
regions = df['Region'].copy()

# handle missing values
X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# ---------------- 格式化特征名称 ----------------
def format_feature_name(name):
    """将特征名称中的下划线替换为空格并适当格式化"""
    replacements = {
        'ABF_Consumption_Percapita': 'ABF Consumption Per Capita',
        'Import_Dependence_Ratio': 'Import Dependence Ratio',
        'Export_Orientation': 'Export Orientation',
        'Price_Change': 'Price Change',
        'Ruminant_Share': 'Ruminant Share',
        'Poultry_Share': 'Poultry Share',
        'Feed_Yield': 'Feed Yield',
        'Feed_Price': 'Feed Price',
        'Stocking_Density': 'Stocking Density',
        'Slaughter_Efficiency': 'Slaughter Efficiency',
        'GDP': 'GDP',
        'SSR': 'SSR',
        'Pr': 'Precipitation',
        'T': 'Temperature'
    }
    return replacements.get(name, name.replace('_', ' ').title())

# ---------------- 按地区分别建模和分析 ----------------
unique_regions = regions.unique()
print("=" * 80)
print(f"按地区分别建模分析 - 目标变量: {TARGET}")
print("=" * 80)

for region in unique_regions:
    print(f"\n\n■■■ 地区: {region} ■■■")
    
    # 筛选该地区的数据
    region_mask = regions == region
    X_region = X[region_mask].copy()
    y_region = y[region_mask].copy()
    
    # 检查数据量是否足够
    if len(X_region) < 10:
        print(f"  数据量不足 ({len(X_region)} 个样本)，跳过该地区")
        continue
    
    print(f"  样本数量: {len(X_region)}")
    
    # 训练测试分割
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X_region, y_region, test_size=0.2, random_state=42
        )
        
        # 训练XGBoost模型
        model = xgb.XGBRegressor(
            random_state=42, tree_method="hist",
            objective="reg:squarederror", eval_metric="rmse",
            n_estimators=500, learning_rate=0.05, max_depth=6
        )
        model.fit(X_train, y_train)
        
        # 计算SHAP值
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        
        # 处理SHAP值（确保是2D数组）
        if isinstance(shap_values, list):
            shap_2d = np.mean(np.abs(np.stack(shap_values, axis=0)), axis=0)
        elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            shap_2d = np.mean(np.abs(shap_values), axis=2)
        else:
            shap_2d = np.abs(shap_values)
        
        # 计算平均|SHAP|值
        mean_abs_shap = np.mean(shap_2d, axis=0)
        
        # 创建特征重要性DataFrame
        importance_df = pd.DataFrame({
            'Feature': FEATURES,
            'Mean_SHAP': mean_abs_shap
        })
        
        # 按重要性排序并取前6
        top6_df = importance_df.sort_values('Mean_SHAP', ascending=False).head(6)
        
        # 打印结果
        print(f"  前6重要特征及Mean |SHAP|值:")
        print("  " + "-" * 60)
        for i, (idx, row) in enumerate(top6_df.iterrows(), 1):
            formatted_name = format_feature_name(row['Feature'])
            print(f"  {i:2d}. {formatted_name:<30} : {row['Mean_SHAP']:.6f}")
        
        # 打印模型性能
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"  模型性能 - 训练集R²: {train_score:.4f}, 测试集R²: {test_score:.4f}")
        
    except Exception as e:
        print(f"  建模过程中出现错误: {str(e)}")
        continue

print("\n" + "=" * 80)
print("所有地区分析完成！")
print("=" * 80)


In [ ]:
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

# ---------------- CONFIG ----------------
DATA_PATH = r"D:\研究\ABF新\8.13ML\ML\生产率及解释变量合并2.xlsx"
TARGET = "kcal_per_kg"   # or "gProtein_per_kg"
FEATURES = [
    "Price_Change","Export_Orientation","Import_Dependence_Ratio",
    "ABF_Consumption_Percapita","SSR","Pr","T","GDP",
    "Ruminant_Share","Poultry_Share","Feed_Yield","Feed_Price",
    "Stocking_Density","Slaughter_Efficiency"
]

# 需要重新优化的地区
REGIONS_TO_REOPTIMIZE = ["EUR", "OCE", "MNA"]

# ---------------- Load Data ----------------
df = pd.read_excel(DATA_PATH)
X = df[FEATURES].copy()
y = df[TARGET].copy()
regions = df['Region'].copy()

# handle missing values
X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# ---------------- 格式化特征名称 ----------------
def format_feature_name(name):
    """将特征名称中的下划线替换为空格并适当格式化"""
    replacements = {
        'ABF_Consumption_Percapita': 'ABF Consumption Per Capita',
        'Import_Dependence_Ratio': 'Import Dependence Ratio',
        'Export_Orientation': 'Export Orientation',
        'Price_Change': 'Price Change',
        'Ruminant_Share': 'Ruminant Share',
        'Poultry_Share': 'Poultry Share',
        'Feed_Yield': 'Feed Yield',
        'Feed_Price': 'Feed Price',
        'Stocking_Density': 'Stocking Density',
        'Slaughter_Efficiency': 'Slaughter Efficiency',
        'GDP': 'GDP',
        'SSR': 'SSR',
        'Pr': 'Precipitation',
        'T': 'Temperature'
    }
    return replacements.get(name, name.replace('_', ' ').title())

def optimize_model_for_region(X_region, y_region, region_name, sample_size):
    """为特定地区优化模型参数"""
    
    # 根据样本量和地区特点调整参数策略
    if region_name == "OCE":
        # OCE地区样本量288但R²为负，需要特别保守的参数
        param_grid = {
            'n_estimators': [50, 100],
            'max_depth': [2, 3],
            'learning_rate': [0.01, 0.05],
            'subsample': [0.7, 0.8],
            'colsample_bytree': [0.7, 0.8],
            'reg_alpha': [0.1, 0.5],
            'reg_lambda': [0.1, 0.5]
        }
    elif sample_size < 100:
        # 小样本策略：简化模型，减少复杂度
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [3, 4],
            'learning_rate': [0.01, 0.05],
            'subsample': [0.8, 0.9],
            'colsample_bytree': [0.8, 0.9]
        }
    elif sample_size < 500:
        # 中等样本策略
        param_grid = {
            'n_estimators': [200, 300, 500],
            'max_depth': [4, 5, 6],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.8, 0.9],
            'colsample_bytree': [0.8, 0.9]
        }
    else:
        # 大样本策略
        param_grid = {
            'n_estimators': [300, 500, 800],
            'max_depth': [5, 6, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
    
    cv_folds = min(5, max(2, sample_size // 20))
    
    # 数据标准化
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_region)
    
    # 基础模型
    base_model = xgb.XGBRegressor(
        random_state=42, 
        tree_method="hist",
        objective="reg:squarederror",
        eval_metric="rmse"
    )
    
    # 网格搜索优化参数
    try:
        grid_search = GridSearchCV(
            base_model, param_grid, 
            cv=cv_folds, 
            scoring='r2',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X_scaled, y_region)
        
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        best_cv_score = grid_search.best_score_
        
        return best_model, best_params, best_cv_score, scaler
        
    except Exception as e:
        print(f"  网格搜索失败，使用保守参数: {str(e)}")
        # 使用保守的默认参数
        if region_name == "OCE":
            conservative_params = {
                'n_estimators': 50,
                'max_depth': 2,
                'learning_rate': 0.01,
                'subsample': 0.7,
                'colsample_bytree': 0.7,
                'reg_alpha': 0.5,
                'reg_lambda': 0.5
            }
        else:
            conservative_params = {
                'n_estimators': 100,
                'max_depth': 3,
                'learning_rate': 0.05,
                'subsample': 0.8,
                'colsample_bytree': 0.8
            }
        model = xgb.XGBRegressor(
            **conservative_params,
            random_state=42,
            tree_method="hist",
            objective="reg:squarederror",
            eval_metric="rmse"
        )
        model.fit(X_region, y_region)
        return model, conservative_params, 0, None

# ---------------- 只重新优化指定地区 ----------------
print("=" * 80)
print(f"重新优化测试集R²偏低的地区 - 目标变量: {TARGET}")
print("=" * 80)

results_summary = []

for region in REGIONS_TO_REOPTIMIZE:
    print(f"\n\n■■■ 重新优化地区: {region} ■■■")
    
    # 筛选该地区的数据
    region_mask = regions == region
    X_region = X[region_mask].copy()
    y_region = y[region_mask].copy()
    
    sample_size = len(X_region)
    print(f"  样本数量: {sample_size}")
    
    try:
        # 优化模型参数
        model, best_params, cv_score, scaler = optimize_model_for_region(
            X_region, y_region, region, sample_size
        )
        
        print(f"  最优参数: {best_params}")
        print(f"  交叉验证R²: {cv_score:.4f}")
        
        # 使用多次随机分割验证模型稳定性
        test_scores = []
        train_scores = []
        shap_importance_list = []
        
        n_repeats = 5 if sample_size > 100 else 3
        
        for i in range(n_repeats):
            # 随机分割
            X_train, X_test, y_train, y_test = train_test_split(
                X_region, y_region, test_size=0.2, random_state=42 + i
            )
            
            # 如果需要标准化
            if scaler is not None:
                X_train_scaled = scaler.transform(X_train)
                X_test_scaled = scaler.transform(X_test)
                model.fit(X_train_scaled, y_train)
                y_pred_test = model.predict(X_test_scaled)
                y_pred_train = model.predict(X_train_scaled)
            else:
                model.fit(X_train, y_train)
                y_pred_test = model.predict(X_test)
                y_pred_train = model.predict(X_train)
            
            # 计算R²
            test_r2 = r2_score(y_test, y_pred_test)
            train_r2 = r2_score(y_train, y_pred_train)
            
            test_scores.append(test_r2)
            train_scores.append(train_r2)
            
            # 计算SHAP值（只在最后一次迭代）
            if i == n_repeats - 1:
                explainer = shap.TreeExplainer(model)
                if scaler is not None:
                    shap_values = explainer.shap_values(X_test_scaled)
                else:
                    shap_values = explainer.shap_values(X_test)
                
                # 处理SHAP值
                if isinstance(shap_values, list):
                    shap_2d = np.mean(np.abs(np.stack(shap_values, axis=0)), axis=0)
                elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                    shap_2d = np.mean(np.abs(shap_values), axis=2)
                else:
                    shap_2d = np.abs(shap_values)
                
                mean_abs_shap = np.mean(shap_2d, axis=0)
                shap_importance_list = mean_abs_shap
        
        # 计算平均性能
        avg_test_r2 = np.mean(test_scores)
        avg_train_r2 = np.mean(train_scores)
        std_test_r2 = np.std(test_scores)
        
        print(f"  模型稳定性验证 ({n_repeats}次随机分割):")
        print(f"  平均训练集R²: {avg_train_r2:.4f}")
        print(f"  平均测试集R²: {avg_test_r2:.4f} (±{std_test_r2:.4f})")
        
        # 检查模型可信度
        if avg_test_r2 < 0.3:
            credibility = "低"
        elif avg_test_r2 < 0.6:
            credibility = "中等"
        else:
            credibility = "高"
        
        print(f"  模型可信度: {credibility}")
        
        # 创建特征重要性DataFrame
        importance_df = pd.DataFrame({
            'Feature': FEATURES,
            'Mean_SHAP': shap_importance_list
        })
        
        # 按重要性排序并取前6
        top6_df = importance_df.sort_values('Mean_SHAP', ascending=False).head(6)
        
        # 打印结果
        print(f"  前6重要特征及Mean |SHAP|值:")
        print("  " + "-" * 60)
        for i, (idx, row) in enumerate(top6_df.iterrows(), 1):
            formatted_name = format_feature_name(row['Feature'])
            print(f"  {i:2d}. {formatted_name:<30} : {row['Mean_SHAP']:.6f}")
        
        # 保存结果摘要
        results_summary.append({
            'Region': region,
            'Sample_Size': sample_size,
            'Avg_Test_R2': avg_test_r2,
            'Std_Test_R2': std_test_r2,
            'Credibility': credibility,
            'Top_Features': top6_df['Feature'].tolist(),
            'Top_SHAP_Values': top6_df['Mean_SHAP'].tolist()
        })
        
    except Exception as e:
        print(f"  建模过程中出现错误: {str(e)}")
        continue

# 打印优化结果摘要
print("\n" + "=" * 80)
print("重新优化地区的结果摘要:")
print("=" * 80)
for result in results_summary:
    print(f"\n地区: {result['Region']}")
    print(f"  样本量: {result['Sample_Size']}, 测试集R²: {result['Avg_Test_R2']:.4f} (±{result['Std_Test_R2']:.4f})")
    print(f"  可信度: {result['Credibility']}")
    print(f"  前3特征: {', '.join([format_feature_name(f) for f in result['Top_Features'][:3]])}")

print("\n" + "=" * 80)
print("指定地区重新优化完成！")
print("=" * 80)